In [11]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0"
import pandas as pd
import numpy as np
import random
import pickle
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from datetime import datetime
from tqdm import tqdm
from sklearn.metrics import r2_score
from collections import OrderedDict

# from ucimlrepo import fetch_ucirepo 

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split

from data.data_loader import EPCDataset, PowerWeatherDatasetWithSeason, PowerWeatherDataset, SingleTermDataset, MultiTermDataset
from models.lstm import LSTMModel
from models.lstm_attention import LSTMWithAttention, BiLSTMWithAttention
from models.gru import GRUModel
from models.utils import create_model, train_for_short_term_forecast, train_for_long_term_forecast, load_model, evaluate_for_short_term_forecast, evaluate_for_long_term_forecast

from explainers.utils import get_explainer

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# short term 
features_for_1h = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temperature',	
    'Humidity',	
]

# long term
features_for_6h = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temp_Min',	'Temp_Max', 'Temp_Avg',	'Temp_Range',
    'Humidity_Min',	'Humidity_Max',	'Humidity_Avg',	'Humidity_Range'
]

file_path_for_1h = 'data/final_data.csv'
file_path_for_6h = 'data/final_data_per_6hr_with_avg_range.csv'

Using device: cuda


In [12]:
dataset_params = {
    # for long-term forecast 

    # 'long_term_length' : 365*4, # 6시간 단위 1년 데이터
    'long_term_length' : 90*24,
    'long_term_pred_length' : 256, # 64, 128, 256
    
    'short_term_length' : 30*24, # 1시간 단위 한달 데이터
    'short_term_pred_length' : 7*24, # 일주일 데이터 예측

    # 2. LSTM, GRU, CNN-LSTM  -> 365*24 / 7*24 (비교용)
    # for Short-term forecast 
    'sequence_length' : 24*30,  # -> 일주일 / 10일 / 15일 / 한달  
    'prediction_length' : 24
}

In [17]:
import re
import json

# 파일 경로
file_path = "important_features_all_samples_for_shortermforecasting.txt"

# 저장할 딕셔너리
important_features_dict = {}

# 정규 표현식 패턴
model_path_pattern = re.compile(r"\./trained_models/(.+)")
feature_pattern = re.compile(r"([\w_]+): ([\d\.]+)")

base_features = [
    'Global_active_power',
    'Global_intensity',
    'Sub_metering_1', 
    'Sub_metering_2', 
    'Sub_metering_3', 
    'Temperature',    
    'Humidity',    
]

is_long_term_forecast = False


# 변수 초기화
current_model = None
current_section = None

if is_long_term_forecast == True:
    # 파일 읽기
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            # 모델 경로 감지 및 모델 키 추출
            match = model_path_pattern.match(line)
            if match:
                current_model = match.group(1)
                important_features_dict[current_model] = {"long": {}, "short": {}}
                continue
    
            # 섹션 감지
            if "Important Long-term Features" in line:
                current_section = "long"
                continue
            elif "Important Short-term Features" in line:
                current_section = "short"
                continue
            
            # Feature 값 추출
            match = feature_pattern.match(line)
            if match and current_model and current_section:
                feature_name, score = match.groups()
                if feature_name in base_features:
                    important_features_dict[current_model][current_section][feature_name] = score
                    # important_features_dict[current_model][current_section].append(feature_name)
                    
else:
    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            
            match = model_path_pattern.match(line)
            if match:
                current_model = match.group(1)
                important_features_dict[current_model] = []
                continue

            # Feature 값 추출
            match = feature_pattern.match(line)
            if match and current_model:
                feature_name, score = match.groups()
                if feature_name in base_features:
                    important_features_dict[current_model].append(feature_name)

In [18]:
important_features_dict

{'LSTM_720_24_256_2.pth': ['Sub_metering_3',
  'Global_intensity',
  'Sub_metering_2',
  'Global_active_power',
  'Humidity',
  'Temperature',
  'Sub_metering_1'],
 'GRU_720_24_256_2.pth': ['Global_intensity',
  'Global_active_power',
  'Sub_metering_3',
  'Sub_metering_2',
  'Humidity',
  'Temperature',
  'Sub_metering_1'],
 'LSTM_720_24_256_3.pth': ['Global_intensity',
  'Global_active_power',
  'Sub_metering_2',
  'Sub_metering_3',
  'Humidity',
  'Temperature',
  'Sub_metering_1'],
 'GRU_720_24_256_3.pth': ['Global_intensity',
  'Global_active_power',
  'Sub_metering_2',
  'Sub_metering_3',
  'Temperature',
  'Humidity',
  'Sub_metering_1'],
 'LSTM_720_24_512_2.pth': ['Sub_metering_2',
  'Global_intensity',
  'Sub_metering_3',
  'Global_active_power',
  'Temperature',
  'Humidity',
  'Sub_metering_1'],
 'GRU_720_24_512_2.pth': ['Sub_metering_2',
  'Temperature',
  'Global_intensity',
  'Humidity',
  'Sub_metering_3',
  'Global_active_power',
  'Sub_metering_1'],
 'GRU_720_24_512_3.

In [19]:
### method 1: long, short 각각 중요도 점수 비율 
def select_features_by_termwise_ratio(scores, threshold_ratio=0.85):
    sorted_feats = sorted(scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(float(v) for _, v in sorted_feats)
    selected = []
    running_sum = 0
    for f, v in sorted_feats:
        running_sum += float(v)
        selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected

### method 2: long, short 중요도 합의 점수 비율
def select_features_by_combined_score(long_scores, short_scores, threshold_ratio=0.85):
    all_features = set(long_scores) | set(short_scores)
    combined_scores = {
        f: float(long_scores.get(f, 0)) + float(short_scores.get(f, 0))
        for f in all_features
    }
    sorted_feats = sorted(combined_scores.items(), key=lambda x: float(x[1]), reverse=True)
    total = sum(score for _, score in sorted_feats)
    selected = []
    running_sum = 0
    for f, score in sorted_feats:
        running_sum += score
        selected.append(f)
        if running_sum / total >= threshold_ratio:
            break
    return selected


### method 3: long, short 중요도 순위 합
def select_by_rank_sum(long_ranked: list, short_ranked: list, max_rank_sum: int = 10):
    selected = []
    for feat in long_ranked:
        if feat in short_ranked:
            long_rank = long_ranked.index(feat) + 1  # 순위는 1부터
            short_rank = short_ranked.index(feat) + 1
            if long_rank + short_rank <= max_rank_sum:
                selected.append(feat)
    return selected


### method 4: long, short 중요도 순위 상대 거리
def select_by_relative_rank_distance(
    base_ranked: list,  # 중심 기준 (long or short)
    reference_ranked: list,  # 상대 비교 term
    top_k: int = 5,
    max_distance: int = 3
):
    selected = []
    for i, feat in enumerate(base_ranked[:top_k]):  # 기준 term에서 상위 top_k만
        if feat in reference_ranked:
            ref_rank = reference_ranked.index(feat)
            if abs(ref_rank - i) <= max_distance:
                selected.append(feat)
    return selected


### method 5: Feature 그룹 (종속/독립 고려)
def select_features_with_group_constraints(importance: dict, max_features: int = 5):
    # 1. 그룹 정의
    dependency_groups = {
        'Global_active_power': 'G1',
        'Global_intensity': 'G1',
        'Sub_metering_1': 'G1',
        'Sub_metering_2': 'G1',
        'Sub_metering_3': 'G1',
        'Temperature': 'G2',
        'Humidity': 'G3'
    }

    # 2. 중요도 Top 5 기준 feature 추출
    # top5_feats = [k for k, _ in sorted(importance.items(), key=lambda x: x[1], reverse=True)[:5]]
    top5_feats = importance[:5]

    selected = []
    g1_count = 0

    # 3. Temperature / Humidity 처리
    include_temp = 'Temperature' in top5_feats
    include_hum = 'Humidity' in top5_feats

    if include_temp:
        selected.append('Temperature')
    if include_hum:
        selected.append('Humidity')

    # 4. G1 선택 기준 계산
    max_g1 = max_features - 2  # 최대 G1 선택 수

    # 5. 입력 순서 기준 순회하며 선택
    for feat in importance:
        group = dependency_groups.get(feat)
        if feat in selected:
            continue  # 이미 선택된 경우 skip

        if group == 'G1' and g1_count < max_g1:
            selected.append(feat)
            g1_count += 1

    return selected


# extracted_features_dict_1 = {}
# extracted_features_dict_2 = {}
# extracted_features_dict_3 = {}
# extracted_features_dict_4 = {}
extracted_features_dict_5 = {}


for model_name, all_features in important_features_dict.items():
    # print('\n\nall_features: ',json.dumps(all_features, indent=2))

    # ### method 1
    # extracted_features_dict_1[model_name] = {}
    # long_selected = select_features_by_termwise_ratio(all_features['long'])
    # short_selected = select_features_by_termwise_ratio(all_features['short'])
    # extracted_features_dict_1[model_name]['long'] = long_selected
    # extracted_features_dict_1[model_name]['short'] = short_selected
    # # print('\n extracted features: ', len(extracted_features_dict_1[model_name]['long']), len(extracted_features_dict_1[model_name]['short']))
    
    # ### method 2
    # extracted_features_dict_2[model_name] = {}
    # selected_feastures = select_features_by_combined_score(all_features['long'], all_features['short'])
    # extracted_features_dict_2[model_name]['long'] = selected_feastures
    # extracted_features_dict_2[model_name]['short'] = selected_feastures
    # # print('\nselected_feastures: ', len(selected_feastures))


    # ### method 3
    # extracted_features_dict_3[model_name] = {}
    # selected_feastures = select_by_rank_sum(list(all_features['long']), list(all_features['short']))
    # extracted_features_dict_3[model_name]['long'] = selected_feastures
    # extracted_features_dict_3[model_name]['short'] = selected_feastures

    # ### method 4
    # extracted_features_dict_4[model_name] = {}
    # long_selected = select_by_relative_rank_distance(list(all_features['long']), list(all_features['short']))
    # short_selected = select_by_relative_rank_distance(list(all_features['short']), list(all_features['long']))
    # extracted_features_dict_4[model_name]['long'] = long_selected
    # extracted_features_dict_4[model_name]['short'] = short_selected

    ### method 5
    extracted_features_dict_5[model_name] = {}
    selected_features = select_features_with_group_constraints(all_features, 5)
    extracted_features_dict_5[model_name] = selected_features


# extracted_features = [extracted_features_dict_1, extracted_features_dict_2,
#                       extracted_features_dict_3, extracted_features_dict_4]


extracted_features = [extracted_features_dict_5]

In [20]:
extracted_features_dict_5

{'LSTM_720_24_256_2.pth': ['Humidity',
  'Sub_metering_3',
  'Global_intensity',
  'Sub_metering_2'],
 'GRU_720_24_256_2.pth': ['Humidity',
  'Global_intensity',
  'Global_active_power',
  'Sub_metering_3'],
 'LSTM_720_24_256_3.pth': ['Humidity',
  'Global_intensity',
  'Global_active_power',
  'Sub_metering_2'],
 'GRU_720_24_256_3.pth': ['Temperature',
  'Global_intensity',
  'Global_active_power',
  'Sub_metering_2'],
 'LSTM_720_24_512_2.pth': ['Temperature',
  'Sub_metering_2',
  'Global_intensity',
  'Sub_metering_3'],
 'GRU_720_24_512_2.pth': ['Temperature',
  'Humidity',
  'Sub_metering_2',
  'Global_intensity',
  'Sub_metering_3'],
 'GRU_720_24_512_3.pth': ['Humidity',
  'Global_intensity',
  'Global_active_power',
  'Sub_metering_2'],
 'LSTM_720_24_512_3.pth': ['Humidity',
  'Global_active_power',
  'Global_intensity',
  'Sub_metering_3'],
 'GRU_720_24_1024_2.pth': ['Humidity',
  'Global_intensity',
  'Global_active_power',
  'Sub_metering_2'],
 'LSTM_720_24_1024_2.pth': ['Humi

In [7]:
def load_dataset_new_features(params, important_features_for_long, important_features_for_short, is_long_term_forecast=True):
    selected_features = dict()
    
    if is_long_term_forecast:
        dataset_1h_long = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['long_term_length'],  
            prediction_length=params['long_term_pred_length'],
            target_features=important_features_for_long
        )
        train_long, train_targets_long, eval_long, eval_targets_long = dataset_1h_long.load_data()
        
        dataset_1h_short = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['short_term_length'],  
            prediction_length=params['short_term_pred_length'],
            target_features=important_features_for_short
        )
        train_short, train_targets_short, eval_short, eval_targets_short = dataset_1h_short.load_data()
        
        train_data=(train_long, train_short)
        train_targets=(train_targets_long, train_targets_short)
        eval_data=(eval_long, eval_short)
        eval_targets=(eval_targets_long, eval_targets_short)

        selected_features['long'] = dataset_1h_long.selected_features
        selected_features['short'] = dataset_1h_short.selected_features

    else:
        dataset_1h = EPCDataset(
            file_path=file_path_for_1h,
            sequence_length=params['sequence_length'],  
            prediction_length=params['prediction_length'],
            target_features=important_features_for_short
        )
        
        train_sequence, train_targets_sequence, eval_sequence, eval_targets_sequence = dataset_1h.load_data()

        train_data=train_sequence
        train_targets=train_targets_sequence
        eval_data=eval_sequence
        eval_targets=eval_targets_sequence
    
        selected_features['short'] = dataset_1h.selected_features

    return train_data, train_targets, eval_data, eval_targets, selected_features

In [8]:
def load_trained_model(method_num, params, selected_features):
    if params is None:
        return

    # Extract parameters
    model_name = params['model_name']
    hidden_size = params['hidden_size']
    num_layers = params['num_layers']
    dropout = params['dropout']
    num_epochs = params['num_epochs']
    batch_size = params['batch_size']
    learning_rate = params['learning_rate']
    patience = params['patience']
    mse_decay = params['mse_decay']

    # Determine input size and output size
    if is_long_term_forecast == True:
        input_size = {
            'long': len(selected_features['long']),
            'short': len(selected_features['short']),
        }
        output_size = {
            'long': dataset_params['long_term_pred_length'],
            'short': dataset_params['short_term_pred_length'],
        }
    else:
        input_size = {
            'single': len(selected_features['short'])
        }
        output_size = {
            'single': dataset_params['prediction_length']
        }
        

    if mse_decay:
        mse_alpha = 0.3
        mse_beta = 1.0

    else:
        mse_alpha = 1.0
        mse_beta = 1.0

    # Model 생성 
    model = create_model(
        model_name=model_name,
        input_size=input_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        output_size=output_size,
        dropout=dropout, 
        long_term_length=dataset_params['long_term_length'], 
        short_term_length=dataset_params['short_term_length']
    ) 

    if 'LS_CNNLSTM' in model_name:
        model_path = './trained_models/(mf5)_method{}_{}_long_{}_short_{}_{}_{}_{}_alpha_{}_beta_{}.pth'.format(
            method_num+1,
            model_name, dataset_params['long_term_length'], dataset_params['short_term_length'],
            dataset_params['long_term_pred_length'], hidden_size, num_layers, mse_alpha, mse_beta
        )
    else:
        model_path = './trained_models/(mf5)_method{}_{}_{}_{}_{}_{}.pth'.format(
            5,
            model_name, dataset_params['sequence_length'], dataset_params['prediction_length'],
            hidden_size, num_layers
        )
        
    print("Model path:", model_path)

    # Load pre-trained model or train a new one
    new_saved = True
    if os.path.exists(model_path):
        new_saved = False
        print(f"Loading the pre-trained {model_name} model...")
        model.load_state_dict(torch.load(model_path))
        model.to(torch.device('cuda' if torch.cuda.is_available() else 'cpu'))
    else:
        print(f"{model_name} model not found. Training a new model...")
        if 'LS_CNNLSTM' in model_name:
            train_for_long_term_forecast(
                model=model,
                model_name=model_name,
                train_data=train_data,
                train_targets=train_targets,
                eval_data=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience,
                oversample_eval=True, 
                alpha=mse_alpha, 
                beta=mse_beta
            )
        else:
            train_for_short_term_forecast(
                model=model,
                model_name=model_name,
                train_sequences=train_data,
                train_targets=train_targets,
                eval_sequences=eval_data,
                eval_targets=eval_targets,
                model_path=model_path,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
                patience=patience
            )

    return model, new_saved

In [9]:
if is_long_term_forecast == False:
    result_path = 'top_N_results_single_method5.txt'

    for i, important_features_dict in enumerate(extracted_features):

        i += 4
        
        for key, important_features in important_features_dict.items():

            train_data, train_targets, eval_data, eval_targets, selected_features = load_dataset_new_features(
                params=dataset_params,
                important_features_for_long=None,
                important_features_for_short=important_features,
                is_long_term_forecast=is_long_term_forecast
            )

            model_name = key.split('.')[0].split('_')[0]
            hidden_size = int(key.split('.')[0].split('_')[3])
            num_layer = int(key.split('.')[0].split('_')[4])
                
            model_params = {
                'model_name' : model_name, 
                'hidden_size' : hidden_size,
                'num_layers' : num_layer,  # 3
                'dropout' : 0.3,
                'num_epochs' : 150,
                'batch_size' : 256,
                'learning_rate' : 0.001,
                'patience' : 15,
                'mse_decay' : True
            }

    
            if 'model' in locals():
                del model  # 기존 모델 삭제
                torch.cuda.empty_cache()  # GPU 메모리 해제

            # build model 
            model, new_saved = load_trained_model(i, model_params, selected_features)
            
            if new_saved:
                print("Evaluating the model...")
                if is_long_term_forecast:
                    results = evaluate_for_long_term_forecast(
                        model=model,
                        eval_data=eval_data,
                        eval_targets=eval_targets,
                        model_name=model_params['model_name'],
                        batch_size=model_params['batch_size'], 
                        oversample_eval=True 
                    )
                    results = {key: float(value) for key, value in results.items()}
                else:
                    results = evaluate_for_short_term_forecast(
                        model=model,
                        eval_sequences=eval_data,
                        eval_targets=eval_targets,
                        model_name=model_params['model_name'],
                        batch_size=model_params['batch_size']
                    )
        
                with open(result_path, "a") as f:
                    key = '{}_{}_{}_method_{}'.format(model_name, hidden_size, num_layer, 5)
                    f.write("{}\n".format(key))
                    f.write("{}\n".format(results))
                    f.write("\n\n")
            
            del model
            torch.cuda.empty_cache()


Model path: ./trained_models/(mf5)_method5_GRU_720_24_1024_2.pth
Loading the pre-trained GRU model...


/tmp/ipykernel_540782/4066786499.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Model path: ./trained_models/(mf5)_method5_LSTM_720_24_1024_2.pth
Loading the pre-trained LSTM model...


/tmp/ipykernel_540782/4066786499.py:75: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path))


Model path: ./trained_models/(mf5)_method5_GRU_720_24_1024_3.pth
Loading the pre-trained GRU model...


...

Validation loss improved. Model saved at epoch 116


Epoch 131/150: 100%|██████████████████████████████████████████████████████████████████| 94/94 [01:38<00:00,  1.05s/batch, loss=0.0118]


Early stopping applied at epoch 131
Evaluating the model...


/archive/workspace/XAI/co-work/models/utils.py:329: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  eval_dataset = torch.utils.data.TensorDataset(torch.tensor(eval_sequences, dtype=torch.float32),
/archive/workspace/XAI/co-work/models/utils.py:330: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(eval_targets, dtype=torch.float32))


R² Score: 0.6170
Adjusted R²: 0.6164
SMAPE: 11.72
MASE: 1.6770


In [10]:
print("!")

!
